# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Mã nguồn dưới đây được tối ưu hóa cho Google Colab T4 (16 GB VRAM) và tích hợp sẵn GPU Compatibility Gate.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

import torch
torch.backends.cudnn.benchmark = True

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata, drive

# Mount Drive để lưu adapter và checkpoint đề phòng ngắt session
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount notice:", e)

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception as e:
    print("Colab secrets notice:", e)

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scripts.labcommon import read_jsonl

train_samples = read_jsonl("data/splits/train.jsonl")
val_samples = read_jsonl("data/splits/val.jsonl")
test_samples = read_jsonl("data/splits/test.jsonl")
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

def draw_ground_truth(item):
    img = Image.open(Path("data") / item["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = item["width"], item["height"]
    for obj in item["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline="green", width=2)
        draw.text((x1, max(0, y1 - 10)), obj["label"], fill="green")
    return img

plt.figure(figsize=(12, 4))
for i in range(min(3, len(train_samples))):
    plt.subplot(1, 3, i + 1)
    plt.imshow(draw_ground_truth(train_samples[i]))
    plt.title(f"{train_samples[i]['image_id']}\n({train_samples[i]['session']})")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

def yolo_to_box(label: str, yolo: list) -> dict:
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2) * 1000))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2) * 1000))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2) * 1000))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2) * 1000))))
    if x2 <= x1:
        x2 = min(1000, x1 + 1)
    if y2 <= y1:
        y2 = min(1000, y1 + 1)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects: list) -> str:
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "If no damage is found, output []."
)

# Kiểm tra tính nhất quán
sample_ans = objects_to_answer(train_samples[0]["objects"])
print("Sample JSON answer:", sample_ans)

## 3. E0: zero-shot

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from scripts.labcommon import write_jsonl
from tqdm.auto import tqdm

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

def run_inference(model, samples, out_path, batch_size=8):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    preds = []
    for i in tqdm(range(0, len(samples), batch_size), desc=f"Inference {Path(out_path).stem}"):
        batch = samples[i:i + batch_size]
        images = [Image.open(Path("data") / s["image_path"]).convert("RGB") for s in batch]
        messages = [
            [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
            for img in images
        ]
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to("cuda")
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        prompt_len = inputs["input_ids"].shape[1]
        raw_outputs = processor.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
        for s, raw in zip(batch, raw_outputs):
            preds.append({"image_id": s["image_id"], "raw": raw.strip()})
    write_jsonl(out_path, preds)

model_e0 = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

run_inference(model_e0, test_samples, "work/e0.jsonl", batch_size=8)
del model_e0
torch.cuda.empty_cache()

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from torch.utils.data import Dataset, DataLoader

def encode_example(image, prompt, answer, proc):
    user_msg = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    full_msg = user_msg + [{"role": "assistant", "content": answer}]

    prompt_enc = proc.apply_chat_template(
        user_msg, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    )
    full_enc = proc.apply_chat_template(
        full_msg, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt"
    )

    n_prompt = prompt_enc.input_ids.shape[1]
    assert (full_enc.input_ids[0, :n_prompt] == prompt_enc.input_ids[0]).all()

    labels = full_enc.input_ids.clone()
    labels[0, :n_prompt] = -100

    assert "mm_token_type_ids" in full_enc, "Qwen3-VL processor phai tra ve mm_token_type_ids!"
    mm_ids = full_enc["mm_token_type_ids"]
    assert mm_ids.shape == full_enc.input_ids.shape, "mm_token_type_ids khong khop input_ids!"

    return {
        "input_ids": full_enc.input_ids[0],
        "attention_mask": full_enc.attention_mask[0],
        "mm_token_type_ids": mm_ids[0],
        "pixel_values": full_enc.pixel_values,
        "image_grid_thw": full_enc.image_grid_thw,
        "labels": labels[0]
    }

class RoadDamageDataset(Dataset):
    def __init__(self, samples, proc):
        self.samples = samples
        self.proc = proc

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(Path("data") / item["image_path"]).convert("RGB")
        answer = objects_to_answer(item["objects"])
        return encode_example(image, PROMPT, answer, self.proc)

def collate_fn(batch):
    pad_id = processor.tokenizer.pad_token_id or 151643
    input_ids = torch.nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=pad_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    assert all("mm_token_type_ids" in b for b in batch), "Co sample thieu mm_token_type_ids!"
    mm_token_type_ids = torch.nn.utils.rnn.pad_sequence([b["mm_token_type_ids"] for b in batch], batch_first=True, padding_value=0)
    assert mm_token_type_ids.shape == input_ids.shape, "mm_token_type_ids shape mismatch!"

    pixel_values = torch.cat([b["pixel_values"] for b in batch], dim=0)
    image_grid_thw = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "mm_token_type_ids": mm_token_type_ids,
        "labels": labels,
        "pixel_values": pixel_values,
        "image_grid_thw": image_grid_thw
    }

# Kiem tra mask loss
chk = RoadDamageDataset(train_samples[:1], processor)[0]
masked_tokens = chk['labels'][chk['labels'] != -100]
print('Decoded assistant answer:', processor.decode(masked_tokens))


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_name: str):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()

    llm_pattern = r"language_model\..*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
    vit_pattern = r"visual\.blocks\..*\.(qkv|proj|linear_fc1|linear_fc2)"

    if setup_name == "E3":
        target_regex = f"({llm_pattern}|{vit_pattern})"
    else:
        target_regex = llm_pattern

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_regex,
        bias="none",
        task_type="CAUSAL_LM"
    )
    peft_model = get_peft_model(model, lora_config)

    if setup_name in ("E2", "E3") or setup_name.startswith("E4"):
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True

    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"{setup_name} trainable params: {trainable:,}")
    return peft_model


In [ ]:
# === GPU COMPATIBILITY GATE (SMOKE TEST) ===
print("--- RUNNING GPU COMPATIBILITY GATE ---")
from torch.amp import autocast

smoke_base = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
smoke_model = apply_setup(smoke_base, "E2")
for n, p in smoke_model.named_parameters():
    if "visual.merger" in n:
        assert p.dtype == torch.float32, f"{n} phai o float32!"
        assert p.requires_grad, f"{n} phai co requires_grad=True!"

smoke_loader = DataLoader(
    RoadDamageDataset(train_samples[:2], processor),
    batch_size=2,
    collate_fn=collate_fn
)
smoke_batch = next(iter(smoke_loader))
assert "mm_token_type_ids" in smoke_batch, "smoke_batch thieu mm_token_type_ids!"
assert smoke_batch["mm_token_type_ids"].shape == smoke_batch["input_ids"].shape, "Shape mm_token_type_ids khong khop!"
print("Batch keys:", list(smoke_batch.keys()))

smoke_batch = {k: v.to("cuda") for k, v in smoke_batch.items()}
with autocast("cuda", dtype=torch.float16):
    smoke_loss = smoke_model(**smoke_batch).loss

assert torch.isfinite(smoke_loss), "Loss bi NaN/Inf!"
smoke_loss.backward()

assert any(p.grad is not None and torch.isfinite(p.grad).all() for n, p in smoke_model.named_parameters() if "visual.merger" in n), "Projector khong nhan gradient!"
smoke_save_path = Path("work/smoke_projector.pt")
smoke_save_path.parent.mkdir(parents=True, exist_ok=True)
extra_w = {k: v.cpu() for k, v in smoke_model.named_parameters() if "visual.merger" in k and v.requires_grad}
torch.save(extra_w, smoke_save_path)
assert len(torch.load(smoke_save_path)) == len(extra_w), "Loi save/load projector!"
del smoke_model, smoke_base, smoke_loss
torch.cuda.empty_cache()
print(">>> GPU COMPATIBILITY GATE: 100% PASS! <<<")


## 6. Vòng huấn luyện

In [ ]:
import time
import math
from torch.amp import GradScaler, autocast

def train_model(setup_name, samples_train, max_steps=300, lr=2e-4, batch_size=2, grad_accum=4):
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base_model.config.use_cache = False

    model = apply_setup(base_model, setup_name)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Tối ưu DataLoader: num_workers=2, pin_memory=True, persistent_workers=True
    train_loader = DataLoader(
        RoadDamageDataset(samples_train, processor),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    scaler = GradScaler("cuda")

    warmup_steps = int(max_steps * 0.05)
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    model.train()
    step = 0
    pbar = tqdm(total=max_steps, desc=f"Training {setup_name}")
    epoch_iter = iter(train_loader)

    while step < max_steps:
        optimizer.zero_grad()
        for _ in range(grad_accum):
            try:
                batch = next(epoch_iter)
            except StopIteration:
                epoch_iter = iter(train_loader)
                batch = next(epoch_iter)

            batch = {k: v.to("cuda", non_blocking=True) for k, v in batch.items()}
            with autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss / grad_accum

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss encountered!")
            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        step += 1
        pbar.update(1)
        pbar.set_postfix({"loss": f"{loss.item() * grad_accum:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
    pbar.close()

    elapsed_mins = (time.time() - start_time) / 60.0
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    meta = {
        "trainable_params": trainable_params,
        "peak_vram_gb": round(peak_vram, 2),
        "train_minutes": round(elapsed_mins, 2)
    }

    adapter_dir = f"work/{setup_name}_adapter"
    model.save_pretrained(adapter_dir)
    if setup_name in ("E2", "E3"):
        extra = {
            k: v.cpu() for k, v in model.named_parameters()
            if ("visual.merger" in k or "visual.deepstack_merger_list" in k) and v.requires_grad
        }
        torch.save(extra, f"work/{setup_name}_projector.pt")

    Path(f"work/{setup_name}_meta.json").write_text(json.dumps(meta, indent=2))
    print(f"{setup_name} completed in {meta['train_minutes']} min | Peak VRAM: {meta['peak_vram_gb']} GB")

    del model, base_model, optimizer, train_loader
    torch.cuda.empty_cache()
    return meta

## 7. Suy luận và chấm E1–E3

In [ ]:
from peft import PeftModel
import subprocess

def evaluate_and_score(setup_name):
    base = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    model = PeftModel.from_pretrained(base, f"work/{setup_name}_adapter")

    proj_pt = Path(f"work/{setup_name}_projector.pt")
    if proj_pt.exists():
        state_dict = torch.load(proj_pt, map_location="cuda")
        model.load_state_dict(state_dict, strict=False)

    model.eval()
    pred_path = f"work/{setup_name.lower()}.jsonl"
    run_inference(model, test_samples, pred_path, batch_size=8)

    del model, base
    torch.cuda.empty_cache()

    meta = json.loads(Path(f"work/{setup_name}_meta.json").read_text())
    cmd = [
        "python", "scripts/score.py",
        "--pred", pred_path,
        "--name", setup_name,
        "--out", f"report/evidence/{setup_name.lower()}.json",
        "--meta", f"trainable_params={meta['trainable_params']}",
        "--meta", f"peak_vram_gb={meta['peak_vram_gb']}",
        "--meta", f"train_minutes={meta['train_minutes']}"
    ]
    subprocess.run(cmd, check=True)

# Chạy tuần tự E1, E2, E3
for s in ["E1", "E2", "E3"]:
    print(f"=== Bắt đầu {s} ===")
    train_model(s, train_samples)
    evaluate_and_score(s)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# === E4: Ablation study trên data_size (10% vs 100%) ===
import random
random.seed(42)

n_10pct = max(1, int(len(train_samples) * 0.1))
train_10pct = random.sample(train_samples, n_10pct)

print("=== Bắt đầu E4 (10% data) ===")
train_model("E4_data_0.1", train_10pct)

# Suy luận và chấm điểm E4
base_e4 = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={"": 0}
)
model_e4 = PeftModel.from_pretrained(base_e4, "work/E4_data_0.1_adapter")
if Path("work/E4_data_0.1_projector.pt").exists():
    model_e4.load_state_dict(torch.load("work/E4_data_0.1_projector.pt", map_location="cuda"), strict=False)

run_inference(model_e4, test_samples, "work/e4_data_0.1.jsonl", batch_size=8)
del model_e4, base_e4
torch.cuda.empty_cache()

meta_e4 = json.loads(Path("work/E4_data_0.1_meta.json").read_text())
subprocess.run([
    "python", "scripts/score.py",
    "--pred", "work/e4_data_0.1.jsonl",
    "--name", "E4_data_0.1",
    "--out", "report/evidence/e4_data_0.1.json",
    "--meta", f"trainable_params={meta_e4['trainable_params']}",
    "--meta", f"peak_vram_gb={meta_e4['peak_vram_gb']}",
    "--meta", f"train_minutes={meta_e4['train_minutes']}"
], check=True)

In [ ]:
# === E5 (Tuỳ chọn): GRPO với IoU Reward và Unit Test ===
from scripts.score import parse_output, iou_xyxy

def compute_reward(completion: str, gt_boxes: list) -> float:
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0

    total_iou = 0.0
    all_labels = {g["label"] for g in gt_boxes} | {p["label"] for p in pred}
    for lab in all_labels:
        g = [x["bbox"] for x in gt_boxes if x["label"] == lab]
        p = [x["bbox"] for x in pred if x["label"] == lab]
        pairs = sorted(((iou_xyxy(a, b), i, j) for i, a in enumerate(g) for j, b in enumerate(p)), reverse=True)
        used_g, used_p = set(), set()
        for v, i, j in pairs:
            if v < 0.5:
                break
            if i in used_g or j in used_p:
                continue
            used_g.add(i)
            used_p.add(j)
            total_iou += v

    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * (total_iou / denom)

# Kiểm tra 5 ca test của hàm reward
gt_demo = [{"label": "pothole", "bbox": (100.0, 100.0, 200.0, 200.0)}]
assert compute_reward(json.dumps([{"label": "pothole", "bbox_2d": [100, 100, 200, 200]}]), gt_demo) == 1.0
assert compute_reward("invalid json", gt_demo) == 0.0
assert compute_reward(json.dumps([{"label": "crack", "bbox_2d": [100, 100, 200, 200]}]), gt_demo) == 0.2
assert compute_reward(json.dumps([{"label": "pothole", "bbox_2d": [100, 100, 200, 200]}, {"label": "pothole", "bbox_2d": [10, 10, 20, 20]}]), gt_demo) == 0.6
assert compute_reward("[]", []) == 1.0
print("Reward IoU unit tests: 5/5 PASSED")

In [ ]:
# === Trích xuất tối thiểu 6 ảnh minh hoạ Ground Truth vs Prediction ===
from scripts.score import read_jsonl

eval_pred_path = "work/e2.jsonl" if Path("work/e2.jsonl").exists() else "work/e1.jsonl"
if Path(eval_pred_path).exists():
    preds_dict = {r["image_id"]: parse_output(r["raw"]) for r in read_jsonl(eval_pred_path)}
    report_dir = Path("report")
    report_dir.mkdir(parents=True, exist_ok=True)
    test_dict = {t["image_id"]: t for t in test_samples}
    sample_ids = list(test_dict.keys())[:6]

    for idx, img_id in enumerate(sample_ids):
        item = test_dict[img_id]
        img = Image.open(Path("data") / item["image_path"]).convert("RGB")
        draw = ImageDraw.Draw(img)
        w, h = item["width"], item["height"]

        for obj in item["objects"]:
            xc, yc, bw, bh = obj["yolo"]
            x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
            x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
            draw.rectangle([x1, y1, x2, y2], outline="green", width=3)
            draw.text((x1, max(0, y1 - 12)), f"GT: {obj['label']}", fill="green")

        preds = preds_dict.get(img_id) or []
        for p in preds:
            bx1, by1, bx2, by2 = p["bbox"]
            x1, y1 = bx1 / 1000 * w, by1 / 1000 * h
            x2, y2 = bx2 / 1000 * w, by2 / 1000 * h
            draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
            draw.text((x1, min(h - 12, y2 + 2)), f"Pred: {p['label']}", fill="red")

        out_img = report_dir / f"sample_vis_{idx+1}.jpg"
        img.save(out_img)
    print(f"Đã lưu 6 ảnh minh hoạ vào {report_dir}")
else:
    print("Chưa có file predictions để trực quan hoá.")

In [ ]:
# Kiểm tra tính hợp lệ bài nộp
!python scripts/check_submission.py